# 03 — Evaluation, Ensemble & Grad-CAM

**VisionMind — ITAI 1378 Final Project**  
**Author:** Ahmet Burak Solak

We now compare the three trained models:

1. Per-model accuracy on the held-out test set
2. Per-class precision / recall / F1 (confusion matrix)
3. **Soft-vote ensemble** = averaged softmax probabilities
4. **Grad-CAM** explanations on a sample of test images
5. Inference-latency benchmark (CPU + GPU if available)

## 0. Setup

In [ ]:
import sys, time
from pathlib import Path
if '..' not in sys.path:
    sys.path.insert(0, '..')

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

from src.data_processing import CIFAR10_CLASSES, get_dataloaders
from src.inference import load_models, predict
from src.explainability import GradCAM, overlay_heatmap, _prepare_input

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

## 1. Load trained models

In [ ]:
checkpoints = {
    'custom_cnn'  : '../models/custom_cnn_best.pt',
    'resnet18'    : '../models/resnet18_best.pt',
    'mobilenet_v2': '../models/mobilenet_v2_best.pt',
}
models_ = load_models(checkpoints, device=DEVICE)
print('Loaded:', list(models_.keys()))

## 2. Per-model evaluation on the test set

We loop over the test loader once per model, collecting predictions for the confusion matrix and classification report.

In [ ]:
_, _, test_loader = get_dataloaders(
    data_dir='../data', batch_size=256, num_workers=0, image_size=32,
)
_, _, test_loader_224 = get_dataloaders(
    data_dir='../data', batch_size=256, num_workers=0, image_size=224,
)

def evaluate(model, loader, device):
    model.eval()
    all_y, all_p, all_probs = [], [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x)
            probs = F.softmax(logits, dim=1).cpu()
            preds = probs.argmax(dim=1)
            all_y.append(y); all_p.append(preds); all_probs.append(probs)
    return (torch.cat(all_y).numpy(),
            torch.cat(all_p).numpy(),
            torch.cat(all_probs).numpy())

results = {}
for name, m in models_.items():
    loader = test_loader if name == 'custom_cnn' else test_loader_224
    y_true, y_pred, probs = evaluate(m, loader, DEVICE)
    acc = accuracy_score(y_true, y_pred)
    print(f'{name:<14s}  test acc = {acc:.4f}')
    results[name] = {'y_true': y_true, 'y_pred': y_pred, 'probs': probs, 'acc': acc}

## 3. Confusion matrices

In [ ]:
out_dir = Path('../results/visualizations'); out_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 4))
if len(results) == 1:
    axes = [axes]
for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(r['y_true'], r['y_pred'])
    im = ax.imshow(cm, cmap='Blues')
    ax.set_title(f'{name} (acc={r["acc"]:.3f})')
    ax.set_xticks(range(10)); ax.set_yticks(range(10))
    ax.set_xticklabels(CIFAR10_CLASSES, rotation=45, fontsize=7)
    ax.set_yticklabels(CIFAR10_CLASSES, fontsize=7)
    for i in range(10):
        for j in range(10):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=6)
    fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()
fig.savefig(out_dir / 'confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Per-class classification reports

In [ ]:
for name, r in results.items():
    print(f'\n=== {name} ===')
    print(classification_report(r['y_true'], r['y_pred'], target_names=CIFAR10_CLASSES, digits=3))

## 5. Soft-vote ensemble

Average the per-image softmax probabilities of all three models, then take the argmax. The same images are evaluated by all three models — the only complication is that custom_cnn was trained at 32×32 and the others at 224×224. We collect the test labels once, then re-use them.

*Important:* the three loaders iterate the test set in the **same order** (no shuffle for test loaders), so the i-th probability vector across all three results refers to the same image.

In [ ]:
available = [n for n in ('custom_cnn', 'resnet18', 'mobilenet_v2') if n in results]
if len(available) >= 2:
    # All test loaders return labels in the same order
    y_true = results[available[0]]['y_true']
    avg_probs = np.mean([results[n]['probs'] for n in available], axis=0)
    ens_pred = avg_probs.argmax(axis=1)
    ens_acc = accuracy_score(y_true, ens_pred)
    print(f'Soft-vote ensemble accuracy = {ens_acc:.4f}')
    print(classification_report(y_true, ens_pred, target_names=CIFAR10_CLASSES, digits=3))
    results['ensemble'] = {'y_true': y_true, 'y_pred': ens_pred, 'probs': avg_probs, 'acc': ens_acc}
else:
    print('Need at least two trained models for ensembling. Train more first.')

## 6. Comparison bar chart

In [ ]:
names = list(results.keys())
accs  = [results[n]['acc'] for n in names]
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(names, accs, color=['#3b82f6', '#22c55e', '#f59e0b', '#ef4444'][:len(names)])
ax.set_ylim(0, 1.0)
ax.set_ylabel('Test accuracy')
ax.set_title('VisionMind — model comparison on CIFAR-10 test set')
for b, a in zip(bars, accs):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01, f'{a:.3f}',
            ha='center', fontweight='bold')
fig.tight_layout()
fig.savefig(out_dir / 'model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Grad-CAM gallery

We pick a small mix of correctly-classified and mis-classified test images and produce Grad-CAM heatmaps from the strongest single model (ResNet18 by default).

In [ ]:
from PIL import Image
from torchvision import datasets

explain_name = 'resnet18' if 'resnet18' in models_ else next(iter(models_.keys()))
explain_model = models_[explain_name]
image_size = 32 if explain_name == 'custom_cnn' else 224

raw_test = datasets.CIFAR10(root='../data', train=False, download=False)
indices = [3, 17, 42, 88, 123, 200, 333, 555]
fig, axes = plt.subplots(2, len(indices), figsize=(2.2 * len(indices), 5))

cam = GradCAM(explain_model, explain_name)
try:
    for col, idx in enumerate(indices):
        pil_img, true_label = raw_test[idx]
        x, rgb = _prepare_input(pil_img, image_size)
        x = x.to(DEVICE)
        heatmap, pred_idx, conf = cam(x)
        overlay = overlay_heatmap(rgb, heatmap)
        axes[0, col].imshow(rgb); axes[0, col].axis('off')
        axes[0, col].set_title(f'true={CIFAR10_CLASSES[true_label]}', fontsize=9)
        axes[1, col].imshow(overlay); axes[1, col].axis('off')
        ok = '✓' if pred_idx == true_label else '✗'
        axes[1, col].set_title(f'{ok} pred={CIFAR10_CLASSES[pred_idx]}\n({conf*100:.0f}%)', fontsize=9)
finally:
    cam.remove()

fig.suptitle(f'Grad-CAM ({explain_name})', fontweight='bold', fontsize=12)
fig.tight_layout()
fig.savefig(out_dir / 'gradcam_gallery.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Inference-latency benchmark

In [ ]:
def latency(model, image_size, n=200):
    model.eval()
    x = torch.randn(1, 3, image_size, image_size, device=DEVICE)
    # warmup
    for _ in range(10):
        with torch.no_grad():
            _ = model(x)
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    start = time.time()
    for _ in range(n):
        with torch.no_grad():
            _ = model(x)
    if DEVICE == 'cuda':
        torch.cuda.synchronize()
    return (time.time() - start) / n * 1000.0  # ms / image

for name, m in models_.items():
    sz = 32 if name == 'custom_cnn' else 224
    ms = latency(m, sz)
    print(f'{name:<14s}  {ms:6.2f} ms / image  (image_size={sz}, device={DEVICE})')

## Summary

Save final numbers to `results/metrics.txt` so the README can cite them.

In [ ]:
lines = ['VisionMind — final evaluation results', '=' * 40, '']
for n, r in results.items():
    lines.append(f'{n:<14s}  test_acc = {r["acc"]:.4f}')
Path('../results/metrics.txt').write_text('\n'.join(lines) + '\n')
print('\n'.join(lines))